## Rank perturbations by phenotype strength of the MITO-SLOPE in various datasets

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

import os
import pandas as pd
import numpy as np
from utils import standardize_per_catX, handle_nans, subtract_control, find_end_slope2,\
    cohens_d, t_to_z, z_to_p, TwoSampleT2Test
from scipy.stats import ttest_ind

In [2]:
import scipy
scipy.__version__

'1.4.1'

### Set paths

In [3]:
############################################################################################
dataset='taorf'
########################## Project root directory and path to results ########################
home_path="/home/jupyter-mhaghigh@broadinst-ee45a/" #dgx
mito_project_root_dir=home_path+"bucket/projects/2016_08_01_RadialMitochondriaDistribution_donna/"
save_results_dir=mito_project_root_dir+"/workspace/results/"

### Load dataset specific parameters

In [4]:
import yaml

with open("ds_info.yaml") as f:
    cfg = yaml.safe_load(f)

datasets_path = cfg["datasets_path"]

# format only the paths, mutate in place
for name, params in cfg.items():
    if name == "datasets_path":
        continue
    params["profiles_path"] = params["profiles_path"].format(datasets_path=datasets_path)

# drop datasets_path and keep the rest as ds_info_dict
ds_info_dict = {k: v for k, v in cfg.items() if k != "datasets_path"}

### Read preprocessed metadata

In [5]:
##################### Read preprocessed metadata
annot=pd.read_csv(mito_project_root_dir+"/workspace/metadata/preprocessed/annot_"+dataset+'.csv'\
                 ,dtype = {"Metadata_Plate" : str})

# annot

### Load per_site aggregated data and process
  - Read the saved aggregated per site level data


In [6]:
f_substr='MeanFrac'
target_columns=["Cells_RadialDistribution_"+f_substr+"_mito_tubeness_"+str(i)+"of16" for i in range(5,17)]


per_site_profiles_path=mito_project_root_dir+"/workspace/per_site_aggregated_profiles_newpattern_2/"


if dataset=='lincs':
     uncorr_feats_condese=['Nuclei_AreaShape_FormFactor',\
     'Nuclei_AreaShape_Eccentricity',\
     'Cells_AreaShape_Solidity',\
     'Cells_Intensity_MaxIntensity_Mito',\
     'Cells_AreaShape_Eccentricity',\
     'Cytoplasm_AreaShape_MaxFeretDiameter',\
     'Nuclei_Texture_AngularSecondMoment_DNA_8_45']
else:        
    uncorr_feats_condese=pd.read_csv(save_results_dir+'target_pattern_orth_features_lists/fibroblast_derived.csv')['orth_fs'].tolist()

    
target_features_list=ds_info_dict[dataset]["target_features_list"]

cols2remove_lowVars_eachPlate=[]
per_site_df_ls=[]

batches=annot['Batch'].unique()
for b in batches:
    fileNameToSave=per_site_profiles_path+"/"+dataset+"/"+\
    b+"_site_agg_profiles"+".csv.gz"

    if os.path.exists(fileNameToSave):
        per_site_df_b=pd.read_csv(fileNameToSave)
    
        thrsh_std=0.001
        cols2remove_lowVars_eachPlate+=per_site_df_b[uncorr_feats_condese].std()[per_site_df_b[uncorr_feats_condese].std() < thrsh_std].\
        index.tolist()

        per_site_df_ls.append(per_site_df_b)

per_site_df=pd.concat(per_site_df_ls,axis=0,ignore_index=True)

per_site_df, cp_features_analysiss = handle_nans(per_site_df,\
                                        target_columns+uncorr_feats_condese,thrsh_null_ratio=0.05,\
                                        thrsh_std=0.001, fill_na_method='drop-rows');


common_cols_2merge=list(set(annot.columns) & set(per_site_df.columns))
# annot['Metadata_Plate'] = annot['Metadata_Plate'].astype(str)
per_site_df['Metadata_Plate'] = per_site_df['Metadata_Plate'].astype(str)
# per_site_df['Metadata_Batch'] = per_site_df['Metadata_Batch'].astype(str)
# per_site_df['Metadata_Well'] = per_site_df['Metadata_Well'].astype(str)

if (dataset=="jump_crispr") or (dataset=="jump_compound"):
    merge_how='inner'
    
else:
    merge_how='left'

per_site_df=pd.merge(per_site_df, annot, how=merge_how,on=common_cols_2merge)

if 'Metadata_pert_type' in per_site_df.columns:
    per_site_df=per_site_df[~per_site_df['Metadata_pert_type'].isnull()].reset_index(drop=True)

uncorr_feats_cond = list(set(uncorr_feats_condese) - set(cols2remove_lowVars_eachPlate))

# ---------------------------------------------------
per_site_df = standardize_per_catX(
    per_site_df, "batch_plate", target_columns+uncorr_feats_cond
).copy()


control_df_perplate = (
    per_site_df.loc[per_site_df["ctrl_well"]]
    .groupby(["batch_plate"])[target_columns].mean()
)
plates_with_controls=list(set(per_site_df['batch_plate'].unique().tolist()) & set(control_df_perplate.index.unique().tolist()))

per_site_df=per_site_df[per_site_df['batch_plate'].isin(plates_with_controls)].reset_index(drop=True)

df_rep_level_scaled_meanSub = per_site_df.groupby(
    "batch_plate"
)[target_columns].apply(subtract_control, control_df_perplate)

peak_slope=np.apply_along_axis(find_end_slope2, 1, df_rep_level_scaled_meanSub.values)

    
per_site_df[["last_peak_loc","slope"]] = peak_slope


per_site_df = standardize_per_catX(
    per_site_df, "batch_plate", target_columns+uncorr_feats_cond+["last_peak_loc","slope"]
).copy()

cp_features: 35
cols2remove_manyNulls []
cols2remove_lowVars []
len cp_features_analysis/nan cols/low vars: 35 0 0
before dropping nan rows:  (17243, 44)
after dropping nan rows:  (17243, 44)


### Statistical test between the target pattern for each pert versus controls 

In [7]:
pert_col=ds_info_dict[dataset]["pert_col"]
meta_cols=ds_info_dict[dataset]["meta_cols"]


results=annot[meta_cols].drop_duplicates().reset_index(drop=True)

feature_list2=per_site_df.columns[per_site_df.columns.str.contains("Cells_|Nuclei_|Cytoplasm_")].tolist()


if 'Metadata_pert_type' in per_site_df.columns:
    perts=per_site_df[per_site_df['Metadata_pert_type'].isin(['trt','Treated'])][pert_col].unique()
    
else:
    perts=per_site_df[~per_site_df['ctrl_well']][pert_col].unique()

for peri, pert in enumerate(perts):
    if peri % 100 ==0:
        print(peri,'/',len(perts))
    
    per_site_df_pert=per_site_df[per_site_df[pert_col]==pert].reset_index(drop=True)
    if not per_site_df_pert.empty:
        plates_pert=per_site_df_pert.groupby(['batch_plate']).filter(lambda x: len(x) > 1)['batch_plate'].unique()
#         plates_pert=per_site_df_pert['batch_plate'].unique()

        if len(plates_pert)>0:
            pert_cell_count_perSite_all_plates=[]
            pert_pvals_all_plates=np.full((len(plates_pert),6),np.nan)
            pert_tvals_all_plates=np.full((len(plates_pert),4),np.nan)    
            peak_slope_all_plates=np.full((len(plates_pert),2),np.nan)    
            for pi,plate in enumerate(plates_pert):
                per_site_df_pert_plate=per_site_df_pert[per_site_df_pert['batch_plate']==plate].reset_index(drop=True)

                pert_cell_count_perSite_all_plates.append(per_site_df_pert_plate['Count_Cells'].mean())


                control_df=per_site_df[(per_site_df['ctrl_well']) & \
                            (per_site_df['batch_plate']==plate)].reset_index(drop=True)


                test_res=ttest_ind(per_site_df_pert_plate['slope'], control_df['slope'], equal_var=False)
                cohend = cohens_d(per_site_df_pert_plate['slope'], control_df['slope'])

                pert_tvals_all_plates[pi,3]=cohend

                degfree = per_site_df_pert_plate['slope'].shape[0] + control_df['slope'].shape[0]  - 2
                z_score = t_to_z(test_res.statistic, degfree)
                std_p_val = z_to_p(z_score)
                pert_pvals_all_plates[pi,3]=std_p_val


                pert_pvals_all_plates[pi,2]=test_res.pvalue
                pert_tvals_all_plates[pi,2]=test_res.statistic

                statistic, p_value, p_value_std_pattern = TwoSampleT2Test(control_df[target_columns], per_site_df_pert_plate[target_columns])

                pert_pvals_all_plates[pi,0]=p_value
                pert_tvals_all_plates[pi,0]=statistic
                pert_pvals_all_plates[pi,4]=p_value_std_pattern


                statistic, p_value, p_value_std_orth = TwoSampleT2Test(control_df[uncorr_feats_cond], per_site_df_pert_plate[uncorr_feats_cond])
                pert_pvals_all_plates[pi,1]=p_value
                pert_tvals_all_plates[pi,1]=statistic
                pert_pvals_all_plates[pi,5]=p_value_std_orth


                peak_slope_all_plates[pi,:]=per_site_df_pert_plate[["last_peak_loc","slope"]].median()

            med_t=np.nanpercentile(pert_tvals_all_plates,50,axis=0,interpolation="nearest")

            median_selection_ind=3
            if ~np.isnan(med_t[median_selection_ind]):
                median_t_indx_val= np.argwhere(pert_tvals_all_plates[:,median_selection_ind]==\
                                                med_t[median_selection_ind])[0][0]
                median_t_indx= [median_t_indx_val]*4
                median_t_indx_p= [median_t_indx_val]*6
            else:
                median_t_indx= [np.nan]*4
                median_t_indx_p= [np.nan]*6

            results.loc[results[pert_col]==pert,'Count_Cells_avg']=np.mean(pert_cell_count_perSite_all_plates)
        #     [pert_cell_count_perSite_all_plates[median_t_indx[i],i] for i in range(len(feature_list2))]
            results.loc[results[pert_col]==pert,['p_target_pattern','p_orth','p_slope','p_slope_std',\
                                                 'p_pattern_std','p_orth_std']]=\
            [pert_pvals_all_plates[median_t_indx_p[i],i] for i in range(6)]
            results.loc[results[pert_col]==pert,['t_target_pattern','t_orth','t_slope','d_slope']]=\
            [pert_tvals_all_plates[median_t_indx[i],i] if ~np.isnan(med_t[i]) else np.nan for i in range(4)]

            results.loc[results[pert_col]==pert,['last_peak_ind','slope']]=np.nanmedian(peak_slope_all_plates,axis=0)
    
# write_res_path='../results/'
# results.sort_values(by=['slope'],ascending=False).to_csv(write_res_path+"/"+dataset+"_results_pattern_.csv",index=False)

0 / 323
100 / 323
200 / 323
300 / 323


In [13]:
# results.sort_values(by='Metadata_gene_name')

In [14]:
# pd.read_csv(save_results_dir+"/virtual_screen/"+dataset+"_results_pattern_aug_070624.csv").sort_values(by='Metadata_gene_name')